# 03 — Prepare Visualisation Data
## RCT World Explorer

This notebook takes the cleaned dataset (`rct_cleaned_data.csv`) and produces a lean JSON file (`rct_viz_data.json`) for the static GitHub Pages web app.

### What this notebook does
1. Loads the cleaned data
2. Selects only the columns needed by the viz
3. Splits into two datasets: `country_studies` and `private_studies`
4. Pre-computes country-level aggregates for the map (RCT counts per country)
5. Extracts the unique keyword/sector list for the filter UI
6. Saves everything into a single structured JSON file

### Why a single JSON file?
The web app is a static site hosted on GitHub Pages — no backend. All data must be bundled with the app. A single JSON keeps the fetch to one request on page load.

### TODO (future)
- Add `publications_count` and `publication_url` columns once we build the publication matching pipeline (likely using `DOI Number` as the join key against CrossRef or OpenAlex API)

## Step 0 — Imports and paths

In [ ]:
import pandas as pd
import json
from pathlib import Path
from collections import Counter

CLEAN_PATH = Path("../data/processed/rct_cleaned_data.csv")
OUT_PATH   = Path("../data/viz/rct_viz_data.json")

print(f"Clean data exists: {CLEAN_PATH.exists()}")

## Step 1 — Load cleaned data and select columns

We keep only the columns needed by the viz, dropping everything used purely for data processing.

In [ ]:
df = pd.read_csv(CLEAN_PATH)
print(f"Loaded: {df.shape}")

VIZ_COLS = [
    'RCT_ID',
    'RCT_ID_duplicates',
    'Title',
    'Status',
    'Start date',
    'End date',
    'First registered on',
    'country',
    'continent',
    'continent_region',
    'region_detail',
    'is_multi_country',
    'is_online',
    'keywords_clean',
    'Primary Investigator',
    'Url',
    'DOI Number',
]

df_viz = df[VIZ_COLS].copy()

# Rename for cleaner JSON keys (no spaces)
df_viz = df_viz.rename(columns={
    'Start date':          'start_date',
    'End date':            'end_date',
    'First registered on': 'registered_on',
    'Primary Investigator':'investigator',
    'DOI Number':          'doi',
    'Url':                 'url',
    'Title':               'title',
    'Status':              'status',
})

print(f"Viz columns: {df_viz.columns.tolist()}")
print(f"Shape: {df_viz.shape}")
df_viz.head(3)

## Step 2 — Split into country studies and private studies

- **country_studies**: all rows where `country != 'Private'`
- **private_studies**: all rows where `country == 'Private'`

These map to the two modes in the web app: `COUNTRY_STUDIES` and `PRIVATE_STUDIES`.

In [ ]:
df_country = df_viz[df_viz['country'] != 'Private'].copy()
df_private = df_viz[df_viz['country'] == 'Private'].copy()

print(f"Country studies: {len(df_country)} rows")
print(f"Private studies: {len(df_private)} rows")

# For private studies, country/continent/region_detail are meaningless - drop them
df_private = df_private.drop(columns=['country', 'continent', 'continent_region', 'region_detail'])
print(f"\nPrivate studies columns: {df_private.columns.tolist()}")

## Step 3 — Pre-compute country-level aggregates for the map

The choropleth map needs to know how many RCTs each country has. Rather than computing this in the browser on every filter change, we pre-compute the base counts here.

The browser will handle filtered counts dynamically — this is just the unfiltered baseline.

In [ ]:
# Total RCT count per country (unique RCT_IDs, not rows)
# Note: a multi-country RCT counts once per country it appears in - this is correct
# because we want to show "how many RCTs touched this country"
country_counts = (
    df_country
    .groupby('country')
    .agg(
        rct_count       = ('RCT_ID', 'nunique'),
        completed_count = ('status', lambda x: (x == 'completed').sum()),
        ongoing_count   = ('status', lambda x: (x == 'on_going').sum()),
        continent       = ('continent', 'first'),
        continent_region= ('continent_region', 'first'),
    )
    .reset_index()
    .sort_values('rct_count', ascending=False)
)

print(f"Countries with RCTs: {len(country_counts)}")
print("\nTop 15 countries:")
print(country_counts.head(15).to_string(index=False))

## Step 4 — Extract keyword/sector list for filter UI

We extract all unique keywords from `keywords_clean` and count their frequency. This powers the sector filter dropdown in the web app.

We separate the AEA's **top-level sector tags** (single words like `education`, `health`, `labor`) from free-text keywords, as these make the most useful filter options.

In [ ]:
# AEA top-level sector tags (these appear very frequently and are standardised)
AEA_SECTORS = {
    'education', 'health', 'labor', 'behavior', 'finance', 'governance',
    'welfare', 'agriculture', 'environment_and_energy', 'firms_and_productivity',
    'electoral', 'crime_violence_and_conflict', 'gender', 'other'
}

# Count all keywords across country studies
all_keywords = []
for kws in df_country['keywords_clean'].dropna():
    if kws:
        all_keywords.extend(kws.split('|'))

kw_counts = Counter(all_keywords)

# Split into sectors and free-text keywords
sectors = {k: v for k, v in kw_counts.items() if k in AEA_SECTORS}
free_keywords = {k: v for k, v in kw_counts.items() 
                 if k not in AEA_SECTORS and v >= 5}  # min 5 occurrences

print(f"AEA sector tags: {len(sectors)}")
print("\nSectors with counts:")
for k, v in sorted(sectors.items(), key=lambda x: -x[1]):
    print(f"  {k}: {v}")

print(f"\nFree-text keywords (freq >= 5): {len(free_keywords)}")
print("\nTop 20 free-text keywords:")
for k, v in sorted(free_keywords.items(), key=lambda x: -x[1])[:20]:
    print(f"  {k}: {v}")

## Step 5 — Format records for JSON

We convert each dataframe to a list of dictionaries. A few things to handle:
- Boolean columns (`is_multi_country`, `is_online`) must be Python bools not numpy bools
- NaN values must be converted to `null` (JSON-compatible)
- Keywords stay as pipe-separated strings — the browser splits them on `|`

In [ ]:
def df_to_records(df):
    """Convert dataframe to JSON-serialisable list of dicts."""
    records = []
    for _, row in df.iterrows():
        record = {}
        for col, val in row.items():
            if pd.isna(val) if not isinstance(val, (list, dict)) else False:
                record[col] = None
            elif isinstance(val, (bool, )):
                record[col] = bool(val)
            elif hasattr(val, 'item'):  # numpy scalar
                record[col] = val.item()
            else:
                record[col] = val
        records.append(record)
    return records

country_records = df_to_records(df_country)
private_records = df_to_records(df_private)
country_agg     = df_to_records(country_counts)

print(f"Country records: {len(country_records)}")
print(f"Private records: {len(private_records)}")
print(f"Country aggregates: {len(country_agg)}")
print(f"\nSample country record:")
print(json.dumps(country_records[0], indent=2))

## Step 6 — Assemble and save JSON

We bundle everything into a single JSON object:

```json
{
  "meta": { ... },
  "country_studies": [ ... ],
  "private_studies": [ ... ],
  "country_aggregates": [ ... ],
  "sectors": [ ... ],
  "keywords": [ ... ]
}
```

The `meta` block gives the app key stats to display in the UI without having to compute them from the records.

In [ ]:
output = {
    "meta": {
        "total_rcts":           df['RCT_ID'].nunique(),
        "country_study_rcts":   df_country['RCT_ID'].nunique(),
        "private_study_rcts":   df_private['RCT_ID'].nunique(),
        "total_countries":      df_country['country'].nunique(),
        "last_updated":         pd.Timestamp.now().strftime('%Y-%m-%d'),
    },
    "country_aggregates": country_agg,
    "country_studies":    country_records,
    "private_studies":    private_records,
    "sectors": [
        {"name": k, "count": v}
        for k, v in sorted(sectors.items(), key=lambda x: -x[1])
    ],
    "keywords": [
        {"name": k, "count": v}
        for k, v in sorted(free_keywords.items(), key=lambda x: -x[1])
    ],
}

# Save
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(OUT_PATH, 'w') as f:
    json.dump(output, f, separators=(',', ':'))  # compact, no whitespace

file_size_kb = OUT_PATH.stat().st_size / 1024
print(f"Saved: {OUT_PATH}")
print(f"File size: {file_size_kb:.1f} KB")
print(f"\nMeta block:")
print(json.dumps(output['meta'], indent=2))

## Step 7 — Validate JSON

Quick sanity checks to confirm the JSON is complete and well-formed before the web app uses it.

In [ ]:
with open(OUT_PATH) as f:
    data = json.load(f)

print("=== VALIDATION ===")
print(f"Keys: {list(data.keys())}")
print(f"\nMeta: {data['meta']}")
print(f"\nCountry aggregates: {len(data['country_aggregates'])} countries")
print(f"Country studies:    {len(data['country_studies'])} rows")
print(f"Private studies:    {len(data['private_studies'])} rows")
print(f"Sectors:            {len(data['sectors'])}")
print(f"Keywords:           {len(data['keywords'])}")

# Check top countries match expectation
print("\nTop 10 countries by RCT count:")
top = sorted(data['country_aggregates'], key=lambda x: -x['rct_count'])[:10]
for c in top:
    print(f"  {c['country']}: {c['rct_count']}")

# Check a sample record has all expected fields
sample = data['country_studies'][0]
print(f"\nSample record keys: {list(sample.keys())}")